# Neural-LAM COSMO sample: end-to-end reproduction

Notebook version of [docs/reproduce_paper_sample.md](../docs/reproduce_paper_sample.md): downloads the
COSMO sample (313 GB) and the published checkpoint, fetches the ERA5 and IFS boundary subsets from
WeatherBench 2, builds the datastores and the graph, trains for one epoch and evaluates the published
checkpoint. Needs one GPU, about 100 GB RAM and 350 GB disk. On CSCS Clariden use
[reproduce_paper_sample_clariden.ipynb](reproduce_paper_sample_clariden.ipynb), which reads the local
copies instead of downloading them.

## Setup (once, in a terminal)

```bash
git clone --branch research https://github.com/joeloskarsson/neural-lam-dev.git && cd neural-lam-dev
uv python install 3.12 && uv venv .venv --python 3.12 && source .venv/bin/activate
uv pip install torch --index-url https://download.pytorch.org/whl/cu126   # match your CUDA version
uv pip install ".[graph]" "zarr<3" ipykernel \
    "mllam-data-prep @ git+https://github.com/sadamov/mllam-data-prep.git@building-ml-lams" \
    "weather-model-graphs @ git+https://github.com/joeloskarsson/weather-model-graphs.git@building-ml-lams"
python -m ipykernel install --user --name neural-lam-sample
```

Then open this notebook with the `neural-lam-sample` kernel. The shell cells below inherit `WORKDIR`,
`SRCDIR` and `PYTHON` from the first code cell.

In [ ]:
import os
import sys
from pathlib import Path

SRCDIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
WORKDIR = Path("/path/to/your/working/directory")  # <--- CHANGE THIS (about 350 GB)
WORKDIR.mkdir(parents=True, exist_ok=True)
os.environ.update(SRCDIR=str(SRCDIR), WORKDIR=str(WORKDIR), PYTHON=sys.executable)
os.environ.setdefault("WANDB_MODE", "offline")  # neural-lam logs to wandb

## Step 0: COSMO sample and published checkpoint

About 2 h for the 313 GB sample ([ETH Research Collection](https://www.research-collection.ethz.ch/entities/researchdata/3e24bbad-2885-4855-a100-78d18ed321f9)); the checkpoint is [Zenodo record 15131838](https://zenodo.org/records/15131838).

In [ ]:
%%bash
set -e
cd "$WORKDIR"
if [ ! -d cosmo_sample.zarr ]; then
    curl -L -o cosmo_sample.zip "https://libdrive.ethz.ch/index.php/s/OkneRXjZI4eum7e/download"
    unzip cosmo_sample.zip -d cosmo_sample
    mv cosmo_sample/720460/cosmo_sample.zarr cosmo_sample.zarr
    rm -rf cosmo_sample cosmo_sample.zip
fi
[ -f cosmo_model.ckpt ] || curl -L -o cosmo_model.ckpt \
    "https://zenodo.org/api/records/15131838/files/cosmo_model.ckpt/content"
ls -l "$WORKDIR"

## Step 2: sample configs and artifacts

In [ ]:
%%bash
set -e
cd "$WORKDIR"
for f in cosmo_interior_config_sample.yaml cosmo_era5_config_sample.yaml cosmo_ifs_config_sample.yaml \
         cosmo_build_graphs_sample.sh cosmo_model_config_era5_sample.yaml cosmo_model_config_ifs_sample.yaml; do
    cp "$SRCDIR/scripts/$f" .
done
rm -rf cosmo_land_sea_mask.zarr && cp -r "$SRCDIR/scripts/artifacts/cosmo_land_sea_mask.zarr" .
mkdir -p figures && cp "$SRCDIR/figures/earth_texture.jpeg" figures/
# Statistics the published checkpoint was trained with (docs/cosmo_dataset_reference.md); the yaml
# sibling only lends its file stem, its content is not read.
rm -rf cosmo_train_stats.zarr && cp -r "$SRCDIR/scripts/artifacts/cosmo_train_stats.zarr" .
cp cosmo_interior_config_sample.yaml cosmo_train_stats.yaml
sed 's|^  config_path: cosmo_interior_config_sample.yaml$|&\n  overload_stats_path: cosmo_train_stats.yaml|' \
    cosmo_model_config_ifs_sample.yaml > cosmo_model_config_ifs_published.yaml

## Step 3: ERA5 and IFS boundary subsets from WeatherBench 2

About 2 GB and 36 GB, around 30 min.

In [ ]:
%%bash
set -e
cd "$SRCDIR"
[ -d "$WORKDIR/cosmo_era5_sample.zarr" ] || "$PYTHON" scripts/era_download.py \
    --start "2015-12-25T00" --end "2016-03-07T00" --output-name cosmo_era5_sample.zarr --output-dir "$WORKDIR"
[ -d "$WORKDIR/cosmo_ifs_sample.zarr" ] || "$PYTHON" scripts/ifs_download.py \
    --start "2015-12-25T00" --end "2016-03-07T00" --output-name cosmo_ifs_sample.zarr --output-dir "$WORKDIR"

## Step 5: datastores with mllam-data-prep

In [ ]:
%%bash
set -e
cd "$WORKDIR"
"$PYTHON" -m mllam_data_prep cosmo_interior_config_sample.yaml
"$PYTHON" -m mllam_data_prep cosmo_era5_config_sample.yaml
"$PYTHON" -m mllam_data_prep cosmo_ifs_config_sample.yaml

## Step 6: graph with weather-model-graphs

The same hierarchical rectangular graph as the best model of the paper.

In [ ]:
%%bash
set -e
cd "$WORKDIR"
PATH="$(dirname "$PYTHON"):$PATH" bash cosmo_build_graphs_sample.sh

## Step 7: one epoch of training (optional)

Only checks that the training loop runs, about 15 min on one GPU. Fine-tuning (Step 8 of the guide, about 4 h) is left out.

In [ ]:
%%bash
set -e
cd "$WORKDIR"
export CUDA_VISIBLE_DEVICES=0   # one GPU (multi-GPU evaluation skews the metrics)
"$PYTHON" -m neural_lam.train_model --config_path cosmo_model_config_era5_sample.yaml \
    --model hi_lam --graph_name rectangular_hierarchical --hidden_dim 300 --hidden_dim_grid 150 \
    --time_delta_enc_dim 32 --processor_layers 2 --batch_size 1 --min_lr 0.001 --epochs 1 \
    --val_interval 10 --val_steps_to_log 1 2 3 4 --ar_steps_eval 4 --precision bf16-mixed \
    --plot_vars T_2M --num_workers 8 --num_nodes 1

## Step 9: evaluate the published checkpoint

24 hourly steps with IFS boundaries on the sample test period, with the normalisation statistics the checkpoint was trained with (`overload_stats_path`, see [docs/cosmo_dataset_reference.md](../docs/cosmo_dataset_reference.md)); forecasts go to `WORKDIR/cosmo_sample_forecasts.zarr`. This checks the pipeline; the paper's scores need the full setup of `docs/reproduce_paper.md`.

In [ ]:
%%bash
set -e
cd "$WORKDIR"
export CUDA_VISIBLE_DEVICES=0   # one GPU (multi-GPU evaluation skews the metrics)
"$PYTHON" -m neural_lam.train_model --config_path cosmo_model_config_ifs_published.yaml \
    --model hi_lam --epochs 1 --eval test --n_example_pred 1 --ar_steps_eval 24 \
    --val_steps_to_log 1 6 12 18 24 --hidden_dim 300 --hidden_dim_grid 150 --time_delta_enc_dim 32 \
    --processor_layers 2 --num_nodes 1 --batch_size 1 --plot_vars T_2M U_10M --precision bf16-mixed \
    --graph_name rectangular_hierarchical --load cosmo_model.ckpt \
    --save_eval_to_zarr_path cosmo_sample_forecasts.zarr

In [ ]:
import matplotlib.pyplot as plt
import xarray as xr

fc = xr.open_zarr(WORKDIR / "cosmo_sample_forecasts.zarr")["state"].isel(start_time=0)
pred = fc.sel(state_feature="T_2M").isel(elapsed_forecast_duration=-1)
# The zarr's `time` coordinate belongs to the last start time; valid time = start_time + lead.
valid = pred["start_time"].values + pred["elapsed_forecast_duration"].values
truth = xr.open_zarr(WORKDIR / "cosmo_sample.zarr")["T_2M"].sel(time=valid).transpose(*pred.dims)
fig, axes = plt.subplots(1, 3, figsize=(16, 3.8))
kw = dict(origin="lower", cmap="RdYlBu_r", vmin=float(truth.min()), vmax=float(truth.max()))
for ax, field, title in [(axes[0], pred, "neural-LAM T_2M, +24 h"), (axes[1], truth, "COSMO analysis")]:
    fig.colorbar(ax.imshow(field.values.T, **kw), ax=ax, label="K")
    ax.set_title(f"{title}, valid {str(valid)[:16]}")
fig.colorbar(axes[2].imshow((pred - truth).values.T, origin="lower", cmap="RdBu_r", vmin=-5, vmax=5),
             ax=axes[2], label="K")
axes[2].set_title(f"forecast - analysis (RMSE {float(((pred - truth) ** 2).mean() ** 0.5):.2f} K)")
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
fig.tight_layout()

## Next steps

- Full COSMO and DANRA reproduction: [docs/reproduce_paper.md](../docs/reproduce_paper.md).
- Every figure and table of the paper: [verification/README.md](../verification/README.md).